In [3]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from matplotlib.patches import Rectangle
import os

OUTPUT_DIR = "Code Outputs/Study Map Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
def out(n): return os.path.join(OUTPUT_DIR, n)

# Set up the figure and axis
fig = plt.figure(figsize=(10, 12), dpi=1000)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

# Set the map extent slightly larger than the bounding box
ax.set_extent([20, 45, -20, 10], crs=ccrs.PlateCarree())

# Add features
ax.add_feature(cfeature.NaturalEarthFeature('cultural', 'admin_0_countries', '10m',
                                            edgecolor='gray', facecolor='none', 
                                            linewidth=0.5))
ax.coastlines(resolution='10m', color='black', linewidth=0.8)

# Draw all standard lakes
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'lakes', '10m',
                                            edgecolor='none', facecolor='lightblue'))
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'ocean', '10m',
                                            edgecolor='none', facecolor='aliceblue'))

# Highlight the specific East African Rift lakes
target_lakes = ['Victoria', 'Tanganyika', 'Malawi', 'Nyasa', 'Albert', 'Edward', 'Kivu', 'Turkana']
shp_path = shpreader.natural_earth(resolution='10m', category='physical', name='lakes')
reader = shpreader.Reader(shp_path)

for record in reader.records():
    lake_name = record.attributes.get('name', '') or record.attributes.get('name_en', '') or ''
    if any(target in lake_name for target in target_lakes):
        ax.add_geometries([record.geometry], ccrs.PlateCarree(),
                          facecolor='red', edgecolor='darkred', zorder=4)

# Add the Lake Labels
# Format: 'Lake Name': (Longitude, Latitude)
lake_labels = {
    'Victoria': (33.0, -1.0),      
    'Tanganyika': (28.8, -6.5),
    'Malawi': (35.2, -11.5),      
    'Turkana': (36.8, 3.5),    
    'Albert': (30.8, 1.8),     
    'Edward': (28.8, -0.2),     
    'Kivu': (28.5, -2.2)
}

# Loop through and plot each label
for name, (lon, lat) in lake_labels.items():
    ax.text(lon, lat, name, 
            transform=ccrs.PlateCarree(),
            fontsize=11, fontweight='bold',
            ha='center', va='center', zorder=6,
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1.5))

# Define and add bounding box
min_lon, max_lon = 27, 38
min_lat, max_lat = -15, 6

rect = Rectangle((min_lon, min_lat), max_lon - min_lon, max_lat - min_lat,
                 linewidth=2.5, edgecolor='black', facecolor='none',
                 transform=ccrs.PlateCarree(), zorder=5, linestyle='--')
ax.add_patch(rect)

# Add gridlines and coordinate labels
gl = ax.gridlines(draw_labels=True, linestyle=':', color='gray', alpha=0.7)
gl.top_labels = False    
gl.right_labels = False  

# Add title and render
plt.title('East African Rift System Highlighted Lakes\nERA5 Data Bounding Box', fontsize=14, pad=15)
plt.savefig(out("Lakes Figure.svg"))
plt.show()